In [1]:
import subprocess
subprocess.run(["pip", "install", "gradio>=4.0.0", "timm>=0.9.12", "-q"],
               check=True)
print("Ready.")

Ready.


In [2]:
import os
import yaml
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import timm
import numpy as np
import gradio as gr
from pathlib import Path
from PIL import Image

print("Imports OK")
print("Gradio version:", gr.__version__)

C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
Gradio version: 6.14.0


In [3]:
# Run from project root — notebook is at models/demo.ipynb

if Path("config.yaml").exists():
    pass   # already at root
else:
    os.chdir("..")

CKPT_PATH   = Path("models/checkpoints/best_model.pth")
CONFIG_PATH = Path("config.yaml")

for name, path in [("config.yaml", CONFIG_PATH),
                    ("best_model.pth", CKPT_PATH)]:
    status = "OK" if path.exists() else "NOT FOUND"
    print(f"  {name:<20}: {path}  [{status}]")

  config.yaml         : config.yaml  [OK]
  best_model.pth      : models\checkpoints\best_model.pth  [OK]


In [4]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

IMG_SIZE    = cfg["data"]["image_size"]    # 224
NUM_CLASSES = cfg["data"]["num_classes"]   # 6
CLASS_MAP   = cfg["classes"]              # {Plastic:0, ...}
IDX_TO_CLASS = {v: k for k, v in CLASS_MAP.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {DEVICE}")
print(f"Classes     : {list(CLASS_MAP.keys())}")

Device      : cpu
Classes     : ['Plastic', 'Paper', 'Glass', 'Metal', 'Organic', 'Textile']


In [5]:
# Must match training architecture exactly
backbone = timm.create_model(
    "mobilenetv3_large_100",
    pretrained=False,
    num_classes=0,
    global_pool="avg"
)

with torch.no_grad():
    feat_dim = backbone(
        torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
    ).shape[1]

class MobileNetTrash(nn.Module):
    def __init__(self, backbone, feat_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = MobileNetTrash(backbone, feat_dim, NUM_CLASSES)

# Load trained weights
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

print(f"Model loaded — epoch={ckpt['epoch']}  "
      f"val_acc={ckpt['val_acc']:.4f}")
print(f"Feature dim : {feat_dim}")

Model loaded — epoch=9  val_acc=0.8444
Feature dim : 1280


In [6]:
# Same as val_transform in training — no augmentation
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])
print("Transform ready.")

Transform ready.


In [7]:
# Class descriptions shown in the demo UI
CLASS_DESCRIPTIONS = {
    "Plastic"  : "Bottles, bags, containers, straws, cutlery",
    "Paper"    : "Newspaper, cardboard, office paper, magazines",
    "Glass"    : "Bottles, food jars, cosmetic containers",
    "Metal"    : "Aluminum cans, steel cans, aerosol cans",
    "Organic"  : "Food waste, fruit peels, coffee grounds, eggshells",
    "Textile"  : "Clothing, shoes",
}

# Recycling tips shown after prediction
RECYCLING_TIPS = {
    "Plastic"  : "Rinse and place in plastic recycling bin. Remove caps.",
    "Paper"    : "Flatten cardboard. Keep dry — wet paper cannot be recycled.",
    "Glass"    : "Rinse and place in glass bin. Do not mix with ceramics.",
    "Metal"    : "Rinse cans. Aluminium and steel can both be recycled.",
    "Organic"  : "Compost bin or food waste collection. Avoid plastic bags.",
    "Textile"  : "Donate if wearable. Otherwise textile recycling bank.",
}

def classify_image(image):
    """
    Gradio prediction function.
    Input : PIL Image (from Gradio upload)
    Output: (label_dict, tip_string) for Gradio Label + Textbox
    """
    if image is None:
        return {}, "Please upload an image."

    # Preprocess
    tensor = transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)

    # Inference
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx   = int(np.argmax(probs))
    pred_class = IDX_TO_CLASS[pred_idx]
    confidence = float(probs[pred_idx])

    # Build scores dict for Gradio Label component
    scores = {
        f"{IDX_TO_CLASS[i]}": float(probs[i])
        for i in range(NUM_CLASSES)
    }

    # Tip text shown below prediction
    tip = (
        f"Predicted: {pred_class} ({confidence*100:.1f}% confidence)\n"
        f"What it includes: {CLASS_DESCRIPTIONS[pred_class]}\n"
        f"Recycling tip: {RECYCLING_TIPS[pred_class]}"
    )

    return scores, tip

print("classify_image function ready.")

classify_image function ready.


In [8]:
# Example images for the demo — update paths to real images from your dataset
# Place a few sample images in models/demo_samples/ for this to work
SAMPLE_DIR = Path("models/demo_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
examples = [[str(p)] for p in SAMPLE_DIR.glob("*.png")][:6]  # up to 6 samples

with gr.Blocks(theme=gr.themes.Soft(), title="Trash Classifier") as demo:

    gr.Markdown("""
    # 🗑️ Trash Classification System
    **MobileNetV3-Large** trained on the Recyclable and Household Waste Dataset
    Upload an image of waste to classify it into one of 6 categories.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                type="pil",
                label="Upload waste image",
                height=320
            )
            predict_btn = gr.Button(
                "Classify", variant="primary", size="lg"
            )

        with gr.Column(scale=1):
            label_output = gr.Label(
                num_top_classes=6,
                label="Predicted class probabilities"
            )
            tip_output = gr.Textbox(
                label="Result and recycling tip",
                lines=4,
                interactive=False
            )

    # Example images if available
    if examples:
        gr.Examples(
            examples=examples,
            inputs=image_input,
            label="Example images"
        )

    # Model info footer
    gr.Markdown(f"""
    ---
    **Model:** MobileNetV3-Large | **Val accuracy:** {ckpt['val_acc']*100:.1f}%
    | **Classes:** Plastic · Paper · Glass · Metal · Organic · Textile
    """)

    # Wire up button and image change event
    predict_btn.click(
        fn=classify_image,
        inputs=image_input,
        outputs=[label_output, tip_output]
    )
    image_input.change(
        fn=classify_image,
        inputs=image_input,
        outputs=[label_output, tip_output]
    )

print("UI built. Run the next cell to launch.")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22888\2586723587.py:7: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Trash Classifier") as demo:


UI built. Run the next cell to launch.


In [9]:
# share=True creates a public link valid for 72 hours — useful for Kaggle
# share=False for local only

demo.launch(
    share=True,          # set False if running locally and don't need public link
    show_error=True,
    debug=False
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://47979bd003ecb7d03e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
